# RC vs CC estimator selection (NeurIPS 2026 rebuttal)

Do we pick a **different** confidence estimator if we only have **one response per question** (RC)
instead of the full capability target (CC)? And does that choice matter downstream for **pass@k**?

- **Estimators:** raw Verbalized Confidence (`V`), raw P(True) (`T`), in-domain Probe (`P`); set `USE_ISOTONIC=True` to also include isotonic-recalibrated `V+iso`, `T+iso` (5-way selection). No random baseline.
- **CC selection:** m_CC = argmin_m (1/N) sum_i (s_i^m - mu_i)^2, where mu_i = num_correct/num_samples.
- **RC selection:** each of `T=10,000` trials draws one Bernoulli(mu_i) label per question (shared across V/T/P),
  scores each estimator's dataset-level Brier against those labels, and takes the per-trial argmin. RC selection
  probability of a method = fraction of trials it wins.
- **Corrected labels:** targets and pass@k pools come from the canonical `outputs/<split>__<model>/ground_truth.jsonl`
  (this also removes the Olmo verbalized ground-truth contamination).

**Outputs:** `rc_vs_cc_tables.md` (Table 1 + pass@k Tables) and `rc_vs_cc_selection.csv` (full probabilities).

In [7]:
from __future__ import annotations

import csv
import json
from functools import lru_cache
from pathlib import Path

import numpy as np


def find_repo_root() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "neurips2026" / "estimator_results").is_dir():
            return base
    raise FileNotFoundError("no ancestor directory contains neurips2026/estimator_results")


REPO = find_repo_root()
RESULTS = REPO / "neurips2026" / "estimator_results"
OUTPUTS = REPO / "neurips2026" / "outputs"
ANALYSIS = REPO / "analysis"

SEED = 20260729
T_TRIALS = 100000
FLIP_THRESHOLD = 0.05          # Table 1 shows the flip probability only above this
K_VALUES = [1, 4, 16, 64]
N_BOOTSTRAP = 1000

MODELS = [
    ("Olmo-3-7B-Instruct", "Olmo"),
    ("Qwen3-8B-non-thinking", "Qwen3-8B"),
    ("gpt-oss-20b", "gpt-oss-20b"),
]

# dataset label -> (eval_split, calib_split, probe_train)  [same mapping as reliability_diagrams.ipynb]
DATASETS = {
    "TriviaQA": ("triviaqa-validation", "triviaqa-train", "triviaqa-train-cc"),
    "SimpleQA": ("simpleqa-verified", "simpleqa-train", "simpleqa-train-cc"),
    "GSM8K": ("gsm8k-test", "gsm8k-train", "gsm8k-train-cc"),
    "MATH": ("math-500", "math-train", "math-train-cc"),
    "AIME": ("aime-test", "aime-train", "aime-train-cc"),
    "MMLU": ("mmlu-test", "mmlu-validation", "mmlu-validation-cc"),
    "GPQA": ("gpqa-diamond", "gpqa-train", "gpqa-train-cc"),
}

# --- estimator set -------------------------------------------------------------------------------
# USE_ISOTONIC=False -> select among raw {V, T, P}; True -> also add isotonic {V+iso, T+iso}.
USE_ISOTONIC = False
BASE_METHODS = ["V", "T", "P"]
ISO_METHODS = ["Viso", "Tiso"]
METHODS = BASE_METHODS + (ISO_METHODS if USE_ISOTONIC else [])
METHOD_NAME = {"V": "Verbalized", "T": "P(True)", "P": "Probe",
               "Viso": "Verbalized+iso", "Tiso": "P(True)+iso"}
FAMILY = {"V": "V", "Viso": "V", "T": "T", "Tiso": "T", "P": "P"}
MODE_TAG = "iso" if USE_ISOTONIC else "raw"

print(f"repo {REPO}")
print(f"seed={SEED} trials={T_TRIALS} flip_threshold={FLIP_THRESHOLD} k={K_VALUES}")
print(f"USE_ISOTONIC={USE_ISOTONIC} -> methods={METHODS}")

repo /home/sinhan-yang/llm-calibration
seed=20260729 trials=100000 flip_threshold=0.05 k=[1, 4, 16, 64]
USE_ISOTONIC=False -> methods=['V', 'T', 'P']


In [8]:
def read_jsonl(path: Path) -> list[dict]:
    with open(path) as handle:
        return [json.loads(line) for line in handle if line.strip()]


@lru_cache(maxsize=None)
def canonical_gt(eval_split: str, model: str) -> dict[str, tuple[float, int, int]]:
    """example_id -> (mu, num_correct, num_samples) from the model's own 100 samples."""
    out = {}
    for row in read_jsonl(OUTPUTS / f"{eval_split}__{model}" / "ground_truth.jsonl"):
        n, c = int(row["num_samples"]), int(row["num_correct"])
        out[row["example_id"]] = (c / n, c, n)
    return out


def conf_path(method: str, model: str, eval_split: str, probe_train: str) -> Path:
    if method == "P":
        return RESULTS / "linear_probe" / f"{model}__trained-on-{probe_train}" / eval_split / "confidence_predictions.jsonl"
    directory = {"V": "verbalized_confidence", "T": "ptrue"}[method]
    return RESULTS / directory / f"{eval_split}__{model}" / "confidence_predictions_cc.jsonl"


def load_confidence(method: str, model: str, eval_split: str, probe_train: str) -> dict[str, float]:
    rows = read_jsonl(conf_path(method, model, eval_split, probe_train))
    return {r["example_id"]: float(r["confidence"]) for r in rows}


def pool_adjacent_violators(values: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """In-place PAVA: nearest non-decreasing sequence under weighted squared loss."""
    levels, total_weight, span = [], [], []
    for value, weight in zip(values, weights):
        levels.append(float(value)); total_weight.append(float(weight)); span.append(1)
        while len(levels) > 1 and levels[-2] > levels[-1]:
            merged = total_weight[-2] + total_weight[-1]
            levels[-2:] = [(levels[-2] * total_weight[-2] + levels[-1] * total_weight[-1]) / merged]
            total_weight[-2:] = [merged]
            span[-2:] = [span[-2] + span[-1]]
    return np.repeat(levels, span)


def isotonic_fit_predict(x_train, y_train, x_eval) -> np.ndarray:
    """Isotonic regression matching sklearn IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip')."""
    x_train = np.asarray(x_train, float); y_train = np.asarray(y_train, float)
    order = np.argsort(x_train, kind="mergesort")
    knots, inverse = np.unique(x_train[order], return_inverse=True)
    counts = np.bincount(inverse).astype(float)
    means = np.bincount(inverse, weights=y_train[order]) / counts
    fitted = np.clip(pool_adjacent_violators(means, counts), 0.0, 1.0)
    return np.interp(np.asarray(x_eval, float), knots, fitted)


def load_setting(model: str, dataset_label: str) -> dict:
    """Aligned arrays on the common example_id set; targets/pools use corrected canonical labels.

    Isotonic columns (when enabled) are fit on the calibration split's confidences vs the canonical
    target and applied to the eval-split confidences -- the same recipe as the paper's Table 3.
    """
    eval_split, calib_split, probe_train = DATASETS[dataset_label]
    gt = canonical_gt(eval_split, model)
    conf = {m: load_confidence(m, model, eval_split, probe_train) for m in BASE_METHODS}
    ids = sorted(set(gt).intersection(*(conf[m].keys() for m in BASE_METHODS)))
    columns = {m: np.array([conf[m][i] for i in ids], float) for m in BASE_METHODS}

    if USE_ISOTONIC:
        calib_gt = canonical_gt(calib_split, model)
        for base, iso in (("V", "Viso"), ("T", "Tiso")):
            calib_conf = load_confidence(base, model, calib_split, probe_train)
            calib_ids = sorted(set(calib_gt).intersection(calib_conf.keys()))
            columns[iso] = isotonic_fit_predict(
                [calib_conf[i] for i in calib_ids],
                [calib_gt[i][0] for i in calib_ids],
                columns[base],
            )

    S = np.column_stack([columns[m] for m in METHODS])
    mu = np.array([gt[i][0] for i in ids], float)
    c = np.array([gt[i][1] for i in ids], int)
    n = np.array([gt[i][2] for i in ids], int)
    return dict(ids=ids, S=S, mu=mu, c=c, n=n, N=len(ids))

In [9]:
def cc_rc_selection(S: np.ndarray, mu: np.ndarray, *, seed: int = SEED,
                    trials: int = T_TRIALS, batch: int = 1000) -> dict:
    """CC = argmin Brier vs mu. RC = per-trial argmin Brier vs one Bernoulli(mu) label per instance.

    Same sampled labels are shared across the three estimators within a trial; exact ties split the
    win equally among tied methods.
    """
    N, M = S.shape
    Lcc = ((S - mu[:, None]) ** 2).mean(axis=0)                 # (M,)
    m_cc = int(np.argmin(Lcc))

    rng = np.random.default_rng(seed)
    counts = np.zeros(M)
    done = 0
    while done < trials:
        b = min(batch, trials - done)
        Y = (rng.random((b, N)) < mu[None, :]).astype(np.float64)      # (b, N) Bernoulli(mu)
        L = ((S[None] - Y[:, :, None]) ** 2).mean(axis=1)              # (b, M) dataset-level Brier
        win = L == L.min(axis=1, keepdims=True)
        counts += (win / win.sum(axis=1, keepdims=True)).sum(axis=0)
        done += b

    probs = counts / trials
    non_cc = [k for k in range(M) if k != m_cc]
    alt = non_cc[int(np.argmax(probs[non_cc]))]
    return dict(Lcc=Lcc, m_cc=m_cc, probs=probs, modal=int(np.argmax(probs)),
                alt=alt, flip=float(1.0 - probs[m_cc]))


RESULTS_T1: dict[tuple[str, str], dict] = {}
for model, _ in MODELS:
    for ds in DATASETS:
        st = load_setting(model, ds)
        sel = cc_rc_selection(st["S"], st["mu"])
        RESULTS_T1[(model, ds)] = {**st, **sel}
        probs = {m: round(float(p), 3) for m, p in zip(METHODS, sel["probs"])}
        print(f"{ds:9s} {model:22s} N={st['N']:4d} CC={METHODS[sel['m_cc']]} "
              f"RCprobs={probs} flip={sel['flip'] * 100:5.1f}%")

TriviaQA  Olmo-3-7B-Instruct     N=1000 CC=P RCprobs={'V': 0.0, 'T': 0.0, 'P': 1.0} flip=  0.0%
SimpleQA  Olmo-3-7B-Instruct     N=1000 CC=P RCprobs={'V': 0.0, 'T': 0.0, 'P': 1.0} flip=  0.0%
GSM8K     Olmo-3-7B-Instruct     N=1319 CC=P RCprobs={'V': 0.0, 'T': 0.0, 'P': 1.0} flip=  0.0%
MATH      Olmo-3-7B-Instruct     N= 500 CC=P RCprobs={'V': 0.0, 'T': 0.0, 'P': 1.0} flip=  0.0%
AIME      Olmo-3-7B-Instruct     N=  90 CC=P RCprobs={'V': 0.0, 'T': 0.0, 'P': 1.0} flip=  0.0%
MMLU      Olmo-3-7B-Instruct     N=1000 CC=P RCprobs={'V': 0.0, 'T': 0.0, 'P': 1.0} flip=  0.0%
GPQA      Olmo-3-7B-Instruct     N= 198 CC=P RCprobs={'V': 0.0, 'T': 0.0, 'P': 1.0} flip=  0.0%
TriviaQA  Qwen3-8B-non-thinking  N=1000 CC=P RCprobs={'V': 0.0, 'T': 0.0, 'P': 1.0} flip=  0.0%
SimpleQA  Qwen3-8B-non-thinking  N=1000 CC=P RCprobs={'V': 0.0, 'T': 0.0, 'P': 1.0} flip=  0.0%
GSM8K     Qwen3-8B-non-thinking  N=1319 CC=P RCprobs={'V': 0.0, 'T': 0.0, 'P': 1.0} flip=  0.0%
MATH      Qwen3-8B-non-thinking  N= 500 

In [10]:
def table1_cell(r: dict) -> str:
    """`CC;RC`. When flip prob >= threshold, show the RC alternative and the flip probability.

    Bold marks a cross-family swap (V/V+iso vs T/T+iso vs P) -- a genuinely different estimator,
    not just a recalibration of the same one."""
    cc = METHODS[r["m_cc"]]
    if r["flip"] < FLIP_THRESHOLD:
        return f"{cc};{cc}"
    alt = METHODS[r["alt"]]
    text = f"{cc};{alt} ({r['flip'] * 100:.1f}%)"
    return f"**{text}**" if FAMILY[cc] != FAMILY[alt] else text


def render_table1() -> str:
    header = "| CC;RC (flip%) | " + " | ".join(DATASETS) + " |"
    sep = "|" + "---|" * (len(DATASETS) + 1)
    lines = [header, sep]
    for model, label in MODELS:
        cells = " | ".join(table1_cell(RESULTS_T1[(model, ds)]) for ds in DATASETS)
        lines.append(f"| **{label}** | {cells} |")
    return "\n".join(lines)


table1 = render_table1()
print(table1)

| CC;RC (flip%) | TriviaQA | SimpleQA | GSM8K | MATH | AIME | MMLU | GPQA |
|---|---|---|---|---|---|---|---|
| **Olmo** | P;P | P;P | P;P | P;P | P;P | P;P | P;P |
| **Qwen3-8B** | P;P | P;P | P;P | P;P | P;P | P;P | P;P |
| **gpt-oss-20b** | P;P | P;P | V;V | **V;P (20.6%)** | **V;P (6.0%)** | V;V | V;V |


In [11]:
def pass_at_k(n: int, c: int, k: int) -> float:
    """Unbiased pass@k estimator (Chen et al. 2021); matches passk_simulation/src/passk_simulator.py."""
    if n - c < k:
        return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))


def passk_squared_error(conf: np.ndarray, c: np.ndarray, n: np.ndarray, k: int) -> np.ndarray:
    """Per-query squared error between predicted pass@k (1-(1-p)^k) and the unbiased actual pass@k."""
    predicted = 1.0 - (1.0 - conf) ** k
    actual = np.array([pass_at_k(int(nn), int(cc), k) for nn, cc in zip(n, c)])
    return (actual - predicted) ** 2


def passk_experiment(dataset_label: str) -> tuple[list, dict]:
    """Two rows per model (CC-selected, RC alternative) + paired bootstrap CI of MSE_alt - MSE_cc."""
    rows, boot = [], {}
    for model, label in MODELS:
        r = RESULTS_T1[(model, dataset_label)]
        S, c, n = r["S"], r["c"], r["n"]
        cc_i, alt_i = r["m_cc"], r["alt"]
        se_cc = {k: passk_squared_error(S[:, cc_i], c, n, k) for k in K_VALUES}
        se_alt = {k: passk_squared_error(S[:, alt_i], c, n, k) for k in K_VALUES}

        rows.append((label, METHOD_NAME[METHODS[cc_i]], "CC-selected", None,
                     {k: float(se_cc[k].mean()) for k in K_VALUES}))
        rows.append(("", METHOD_NAME[METHODS[alt_i]], "RC alternative", float(r["probs"][alt_i]),
                     {k: float(se_alt[k].mean()) for k in K_VALUES}))

        rng = np.random.default_rng(SEED)
        N = r["N"]
        samples = {k: np.empty(N_BOOTSTRAP) for k in K_VALUES}
        for b in range(N_BOOTSTRAP):
            idx = rng.integers(0, N, N)
            for k in K_VALUES:
                samples[k][b] = se_alt[k][idx].mean() - se_cc[k][idx].mean()
        boot[label] = {k: (float(np.percentile(samples[k], 2.5)),
                           float(np.percentile(samples[k], 97.5))) for k in K_VALUES}
    return rows, boot


def render_passk(dataset_label: str) -> str:
    rows, boot = passk_experiment(dataset_label)
    head = "| Model | Method | RC selection probability | " + " | ".join(f"pass@{k}" for k in K_VALUES) + " |"
    sep = "|" + "---|" * (3 + len(K_VALUES))
    out = [f"### {dataset_label} (query-level MSE, lower is better)", head, sep]
    for label, method, role, prob, mses in rows:
        if prob is None:
            prob_str = "\u2014"
        else:
            prob_str = f"{prob * 100:.1f}%" + (" (weak, <5%)" if prob < FLIP_THRESHOLD else "")
        method_str = f"{method} ({role})"
        vals = " | ".join(f"{mses[k]:.4f}" for k in K_VALUES)
        out.append(f"| {label} | {method_str} | {prob_str} | {vals} |")

    out += ["", "95% paired-bootstrap CI of MSE(RC alternative) \u2212 MSE(CC-selected); positive => CC better:", "",
            "| Model | " + " | ".join(f"pass@{k}" for k in K_VALUES) + " |",
            "|" + "---|" * (1 + len(K_VALUES))]
    for label, ci in boot.items():
        out.append(f"| {label} | " + " | ".join(f"[{ci[k][0]:+.4f}, {ci[k][1]:+.4f}]" for k in K_VALUES) + " |")
    return "\n".join(out)


passk_math = render_passk("MATH")
passk_aime = render_passk("AIME")
print(passk_math)
print()
print(passk_aime)

### MATH (query-level MSE, lower is better)
| Model | Method | RC selection probability | pass@1 | pass@4 | pass@16 | pass@64 |
|---|---|---|---|---|---|---|
| Olmo | Probe (CC-selected) | — | 0.0373 | 0.0336 | 0.0219 | 0.0148 |
|  | Verbalized (RC alternative) | 0.0% (weak, <5%) | 0.0632 | 0.0423 | 0.0287 | 0.0208 |
| Qwen3-8B | Probe (CC-selected) | — | 0.0471 | 0.0417 | 0.0265 | 0.0134 |
|  | Verbalized (RC alternative) | 0.0% (weak, <5%) | 0.0977 | 0.0594 | 0.0332 | 0.0185 |
| gpt-oss-20b | Verbalized (CC-selected) | — | 0.0186 | 0.0102 | 0.0067 | 0.0028 |
|  | Probe (RC alternative) | 12.3% | 0.0216 | 0.0102 | 0.0068 | 0.0028 |

95% paired-bootstrap CI of MSE(RC alternative) − MSE(CC-selected); positive => CC better:

| Model | pass@1 | pass@4 | pass@16 | pass@64 |
|---|---|---|---|---|
| Olmo | [+0.0158, +0.0384] | [-0.0003, +0.0194] | [-0.0020, +0.0168] | [-0.0020, +0.0160] |
| Qwen3-8B | [+0.0357, +0.0667] | [+0.0092, +0.0283] | [+0.0017, +0.0143] | [+0.0000, +0.0122] |
| gpt-o

In [12]:
doc = "\n\n".join([
    "# RC vs CC estimator selection",
    "Generated by `analysis/rc_vs_cc.ipynb`. Targets and pass@k pools use the corrected canonical "
    "`outputs/<split>__<model>/ground_truth.jsonl` labels. Isotonic variants (when enabled) are fit "
    "on the calibration split vs the canonical target and applied to the eval split. CC selects by "
    "Brier against mu; RC selects by per-trial Brier against one Bernoulli(mu) label per question "
    f"over {T_TRIALS:,} trials (seed {SEED}). Estimator set: {', '.join(METHODS)}.",
    "## Table 1 -- CC-selected vs single-response RC-selected estimator",
    f"Estimator set: {', '.join(METHODS)} (USE_ISOTONIC={USE_ISOTONIC}). Each cell is `CC;RC`. The "
    f"flip probability P(m_RC != m_CC) is shown only when it reaches {FLIP_THRESHOLD:.0%}; otherwise "
    "the two selections coincide and the cell reads `X;X`. **Bold** marks a cross-family swap "
    "(V / V+iso vs T / T+iso vs P).",
    table1,
    "## Table 2 -- downstream pass@k consequence",
    "For each model: the CC-selected estimator and the most frequent RC alternative, with the "
    "alternative's unconditional RC selection probability. pass@1 MSE equals the CC Brier by "
    "construction, so k>1 carries the signal.",
    passk_math,
    passk_aime,
])
tables_path = ANALYSIS / f"rc_vs_cc_tables_{MODE_TAG}.md"
tables_path.write_text(doc)

csv_path = ANALYSIS / f"rc_vs_cc_selection_{MODE_TAG}.csv"
with open(csv_path, "w", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["model", "dataset", "N", "cc", "modal", "alt", "alt_prob", "flip"]
                    + [f"prob_{m}" for m in METHODS] + [f"Lcc_{m}" for m in METHODS])
    for (model, ds), r in RESULTS_T1.items():
        writer.writerow([model, ds, r["N"], METHODS[r["m_cc"]], METHODS[r["modal"]],
                         METHODS[r["alt"]], round(float(r["probs"][r["alt"]]), 4), round(r["flip"], 4)]
                        + [round(float(p), 4) for p in r["probs"]]
                        + [round(float(v), 4) for v in r["Lcc"]])

print("wrote", tables_path.name, "and", csv_path.name)

wrote rc_vs_cc_tables_raw.md and rc_vs_cc_selection_raw.csv
